[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/floating/floating.ipynb)

# Hydrostatic Stability

This example computes the hydrostatic stability of floating bodies, from small tilts to capsizing: whether the water's push and the body's weight, once the body heels over, turn it back upright or further over. Still water pushes on every wet patch of the body's surface; together those pushes lift the body by the weight of the water it displaces, and when the body heels, the line along which they lift moves sideways, away from the line of its weight.

The computation is written in the projective geometric algebra of three-dimensional space. The water surface is modelled as a plane, the centres of gravity as points, the forces as forques and the turns as twists. The water's push on a patch is the line through it square to its plane, `samples | planes`, sized by the pressure and the area, and the buoyancy is the sum of those forques, a single line of force, with no integral over the body's volume. Its sum with the weight is the righting couple. Over `Tilt`, the space of all small tilts, the same sum is a map from tilts to the change of the net force, and paired with the tilt it is the body's stiffness against tilting, a quadratic form whose eigenvalues over the weight are the metacentric heights.

In the notation of hydrostatics the buoyancy reads as the surface integral $\mathbf F = -\oint p\,\mathbf n\,\mathrm dA$ with the pressure $p = \rho g d$ at the depth $d$, which the divergence theorem turns into the displaced weight $\rho g V$, acting upward through the centre of buoyancy.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import tee

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.fluids.floating import render
from examples.fluids.floating.core import box, wet, wigley
from examples.mechanics.simplex import simplex_inertia_weights

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                 # Scalar+
Plane = ga.gatype.vector()                                  # Vector+
Point = ga.gatype.antivector()                              # Antivector-
Twist = ga.gatype.bivector()                                # Bivector+
Forque = ga.gatype.antibivector()                           # Antibivector-
Motor = ga.gatype.rotor()                                   # Rotor+
# Turns about horizontal lines through the origin: heel and trim.
Tilt = Twist.derive.nondegenerate.derive.containing("z")    # Bivector+
# The energy of a small tilt.
Stiffness = ga.gatype((Scalar, Tilt, Tilt))                 # Scalar+ <- (Tilt+, Tilt+)

# The still water surface, the point at infinity upward, the vertical shift, and the axis of heel through the
# origin, along the body: the meet of the planes y = 0 and z = 0.
water = mv.z                                                # [] Plane+
up = water.dual()                                           # [] Point-
heave = mv.w ^ water                                        # [] Twist+
heel_axis = mv.yz                                           # [] Twist+
# The origin, from which the hull's enclosed volume is summed.
origin = mv.point([0.0, 0.0, 0.0])                          # [] Point-

# Fresh water's weight per cubic metre, in newtons, and gravity.
fluid_weight = 9810.0
gravity = 9.81

## 1. The water's push

Still water pushes on every wet patch of a body's surface, square to the patch and against it, with a pressure equal to the water's weight per unit volume times the depth of the patch. In the algebra a force is a forque, the line it acts along, sized by its strength: the push on a patch is the line through the patch square to its plane, `point | plane`, sized by the pressure and the area. The buoyancy is the sum of those lines over the wet surface, a single line of force; for a closed surface it lifts the body by the weight of the water it displaces, Archimedes' principle, with no integral over the body's volume.

The body is a closed surface of triangles. The hull here is a Wigley hull, the classic test hull of naval architecture, whose half breadth is a parabola along its length times a parabola down its depth; above the waterline its sides rise straight to a flat deck. Each triangle is cut at the waterline into its wet part, the one step outside the algebra. Over each wet part the pressure is linear in the depth, and the sum is an integral computed by an exact quadrature for second moments over a simplex, the same in any number of dimensions: a uniform triangle has the same second moment as three equal point masses inside it, as the [simplex example](../simplex.py) computes and checks against sampling.

In [ ]:
# The hull's length, beam, draft and freeboard, in metres, and the cells its surface is cut into.
length, beam, draft, freeboard = 4.0, 1.2, 0.5, 0.4
n_cells = 8

In [ ]:
# Per triangle, the points halfway between each corner and the centroid, each standing for a third of its area.
rule = 3 * simplex_inertia_weights(3)


def pressing(triangles: Point, surface: Plane) -> tuple[Point, Forque]:
    """Sample points of the wet part of each triangle under each water surface, and the water's push at each
    per unit of pressure: a line through the point, square to the triangle, against its outward normal, with a
    third of the part's area."""
    parts = wet(triangles, surface)                                     # [..., triangles, parts, 3] Point-
    samples = (parts[..., None, :] * rule).sum(axis=-1)                 # [..., triangles, parts, samples] Point-
    # The plane through a part's corners, wound as its triangle, has twice the part's area as its weight. The
    # hull's enclosed volume, signed by the same winding, is positive exactly when those planes face outward:
    # times its sign, every plane faces outward, whichever way the hull is wound.
    volume = (origin & triangles[..., 0] & triangles[..., 1] & triangles[..., 2]).sum(axis=-1)   # [] Scalar+
    planes = (parts[..., 0] & parts[..., 1] & parts[..., 2]) * (volume / volume.norm())   # [..., triangles, parts] Plane+
    return samples, -(samples | planes[..., None]) / 6                  # [..., triangles, parts, samples] Forque-


def buoyancy(triangles: Point, surface: Plane, fluid_weight: float) -> tuple[Forque, Forque]:
    """The water's push on the body below each water surface, the sum of the pressure lines over its wet
    surface, each pressure the fluid's weight times the depth; and how the push grows as the surface rises."""
    samples, lines = pressing(triangles, surface)
    depths = -(surface[..., None, None, None] & samples)                # [..., triangles, parts, samples] Scalar+
    push = (fluid_weight * depths * lines).sum(axis=(-3, -2, -1))       # [...] Forque-
    growth = (fluid_weight * lines).sum(axis=(-3, -2, -1))              # [...] Forque-
    return push, growth


hull = wigley(length, beam, draft, freeboard, n_cells)                  # [triangles, 3] Point-
upright, _ = buoyancy(hull, water, fluid_weight)                        # [] Forque-
# The hull's weight is the water it displaces floating at its waterline.
weight = upright & heave                                                # [] Scalar+

print("water displaced, in cubic metres:", (weight / fluid_weight).to_array().round(4))
print("the smooth hull's, 4 / 9 of length times beam times draft:", round(4 / 9 * length * beam * draft, 4))

## 2. Floating at a heel, and the righting couple

Heeled by an angle, the body floats at the height where its buoyancy carries its weight. Newton's method finds that height: it corrects a guess by the difference between the weight and the upward part of the buoyancy, over how fast the buoyancy grows as the water surface rises, which is the water's weight times the area the surface cuts from the body. The body's weight is a line too, straight down through its centre of gravity. Heeled, the buoyancy and the weight are parallel lines apart, and their sum is a couple, a pure turning effect: the righting couple. Its size over the weight is the righting arm, the distance between the two lines. When the buoyancy rises on the side the body heels towards, the couple turns it back upright.

In [ ]:
# The heights of the hull's centre of gravity over its waterline, from low ballast to a high load.
centre_heights = [-0.2, -0.1, 0.0, 0.1, 0.2]
# Newton steps for the floating height.
n_newton_steps = 6
# The load drawn, the lowest, and the heel the hull is drawn at.
shown_load, shown_heel = 0, 35.0

In [ ]:
def heeled(degrees: np.ndarray) -> Motor:
    """The turns heeling a body by each angle, about its axis of heel."""
    return (mv.yz * (-np.radians(degrees) / 2)).exp()


def afloat(triangles: Point, turns: Motor, weight: Scalar, heights: Scalar, steps: int) -> Scalar:
    """The height of the water surface over the body's origin at which the body, turned by each motor, floats:
    its buoyancy balancing its weight, by Newton's method from the given heights."""
    for _ in range(steps):
        push, growth = buoyancy(triangles, turns << (water - heights * mv.w), fluid_weight)
        lifting = turns << heave                                        # [...] Twist+
        heights = heights + (weight - (push & lifting)) / (growth & lifting)
    return heights                                                      # [...] Scalar+


def placement(turns: Motor, heights: Scalar) -> Motor:
    """The motion taking the body's frame to the world's, the water surface at the world's `water`: the turn,
    then down by the height of the water surface over the body's origin."""
    return (heave * (heights / 2)).exp() * turns                        # [...] Motor+


def forces(triangles: Point, turns: Motor, heights: Scalar, weight: Scalar, centre: Point) -> tuple[Forque, Forque]:
    """The forces on the body, turned by each motor and floating at each height, in the body's frame: its
    buoyancy, and its weight, a line straight down through its centre of gravity. Their sum is the righting
    couple."""
    push, _ = buoyancy(triangles, turns << (water - heights * mv.w), fluid_weight)
    return push, weight * (centre & (turns << -up))                     # [...] Forque-


centres = mv.point([[0, 0, height] for height in centre_heights])       # [loads] Point-
turn = heeled(np.array([shown_heel]))                                   # [1] Motor+
height = afloat(hull, turn, weight, mv.scalar([[0.0]]), n_newton_steps)   # [1] Scalar+
push, _ = forces(hull, turn, height, weight, centres[shown_load])       # [1] Forque-
save_figure(render.draw_ship(hull, placement(turn, height)[0], push[0], centres[shown_load]), "floating_ship");

On the left, the hull heeled by 35°, its wet surface darker with depth. On the right, the same hull seen along its length: the weight in orange, down from the centre of gravity, the buoyancy in blue, up from where its line crosses the water surface, and the righting arm between them in black. The buoyancy has moved towards the side the hull heels to, and the couple turns it back.

## 3. From a small tilt to capsizing

For a small tilt the same sum can be taken over `Tilt`, the space of all small tilts. In the body's frame the water surface turns against the body, deepening each wet point in proportion to its distance from the axis of the tilt, and the weight turns with it; the change of the net force is a map from tilts to lines, and paired with the tilt it is the body's stiffness against tilting, a quadratic form with one eigenvalue against heel and one against trim. Over the weight these are the metacentric heights, the slopes of the righting arm at upright. Large heels need the full computation, one heel at a time.

In [ ]:
# The heels of the righting arm curves, in degrees, from upright to upside down.
heels = np.linspace(0, 180, 91)

In [ ]:
def stiffness(triangles: Point, heights: Scalar, weight: Scalar, centre: Point) -> Stiffness:
    """The energy of a small tilt of the upright body floating at each height: the change of the net force
    under the tilt, the water surface turning against the body and the weight turning with it, paired with
    the tilt."""
    surface = water - heights * mv.w                                    # [...] Plane+
    samples, lines = pressing(triangles, surface)
    # In the body's frame a small tilt turns the water surface, deepening each wet point by its height above the
    # change of the surface, and turns the direction of the weight with it.
    tilted = surface[..., None, None, None].commutator(Tilt)            # [..., 1, 1, 1] Plane+ <- Tilt+
    pushing = (fluid_weight * -(tilted & samples) * lines).sum(axis=(-3, -2, -1))   # [...] Forque- <- Tilt+
    weighing = weight * (centre & (-up).commutator(Tilt))               # [...] Forque- <- Tilt+
    return Tilt & (pushing + weighing)                                  # [...] Scalar+ <- (Tilt+, Tilt+)


# Upright, the hull floats at its waterline, its weight being the water it displaces there.
stiffnesses = stiffness(hull, mv.scalar([0.0]), weight, centres)        # [loads] Scalar+ <- (Tilt+, Tilt+)
turns = heeled(heels)                                                   # [heels] Motor+
# Every load weighs the same, so the hull floats at the same height at each heel, whatever the load.
heights = afloat(hull, turns, weight, mv.scalar(np.zeros((len(heels), 1))), n_newton_steps)   # [heels] Scalar+
push, weighing = forces(hull, turns[:, None], heights[:, None], weight, centres)   # [heels, loads] Forque-, Forque-
couples = push + weighing                                               # [heels, loads] Forque-
save_figure(render.draw_curves(heels, couples, weight, stiffnesses), "floating_righting")

print("metacentric heights against heel and trim, in metres:\n", (stiffnesses.eigvalsh() / weight).to_array().round(3))

Each curve is the righting arm of the hull, in metres, over its heel from upright to upside down, with the centre of gravity at one height: from 20 cm below the waterline, darkest, to 20 cm above it, lightest. The dashed lines are the tangents at upright that the stiffness gives. With its centre of gravity low the hull rights itself from almost any heel. Raised far enough, its metacentric height turns negative: the arm is negative from the start, and the hull capsizes and floats upside down, where the curves return to zero.

In the notation of naval architecture the righting arm is $GZ(\phi)$, the horizontal distance between the centre of gravity $G$ and the line of buoyancy, and its slope at upright the metacentric height $GM = KB + I_{\mathrm{wp}}/V - KG$: the heights of the centre of buoyancy and of gravity over the keel, and the second moment of the waterplane over the displaced volume.

## 4. The floating cube

A cube of uniform density floats with a face up only when it is light or heavy enough: below $(3 - \sqrt{3})/6 \approx 0.21$ of the water's density, or above $(3 + \sqrt{3})/6 \approx 0.79$. In between it rests heeled, and between a quarter and three quarters edge up. The same code, with the cube's twelve triangles as the surface, maps its righting couple over density and heel; where the couple turns from pushing the cube over to pushing it back as the heel grows, the cube rests stably.

In [ ]:
# The cube's densities relative to water, across the map and for the cubes drawn above it, and the heels of the
# map, in degrees, reaching past a quarter turn either side so that its ends are rests like any other.
densities = np.linspace(0.02, 0.98, 49)
shown_densities = [0.1, 0.23, 0.5, 0.77, 0.9]
cube_heels = np.linspace(-10, 100, 56)

In [ ]:
def stable_heels(couples: Forque) -> np.ndarray:
    """For each row of righting couples over the heels of the map, the smallest heel from upright at which the
    cube rests stably: where the couple turns from pushing it over to pushing it back, between the two heels
    around it in proportion to the couples there. The one step outside the algebra."""
    arms = (-(couples & heel_axis)).to_array()                           # [rows, heels]
    below, above = arms[:, :-1], arms[:, 1:]
    share = -below / np.where(above == below, 1, above - below)
    rests = cube_heels[:-1] + share * np.diff(cube_heels)                # [rows, heels - 1]
    stable = (below < 0) & (above >= 0) & (rests > -1e-6)
    return rests[np.arange(len(arms)), np.argmax(stable, axis=1)]


cube = box(np.ones(3))                                                   # [triangles, 3] Point-
weights = mv.scalar((fluid_weight * densities)[:, None, None])           # [densities, 1] Scalar+
turns = heeled(cube_heels)                                               # [heels] Motor+
heights = afloat(cube, turns, weights, mv.scalar(np.zeros((len(densities), len(cube_heels), 1))), n_newton_steps)
push, weighing = forces(cube, turns, heights, weights, mv.w.dual())      # [densities, heels] Forque-, Forque-
cube_couples = push + weighing                                           # [densities, heels] Forque-
rests = stable_heels(cube_couples[np.searchsorted(densities, shown_densities)])   # [shown]
shown_weights = mv.scalar([[fluid_weight * density] for density in shown_densities])   # [shown] Scalar+
shown_turns = heeled(rests)                                              # [shown] Motor+
shown = placement(shown_turns, afloat(cube, shown_turns, shown_weights, mv.scalar(np.zeros((len(rests), 1))), n_newton_steps))
save_figure(render.draw_cubes(densities, cube_heels, cube_couples, fluid_weight, cube, shown, shown_densities, rests),
            "floating_cubes");

The map shows the righting couple over the cube's density, across, and its heel from upright to a quarter turn, up: blue where it turns the cube back towards upright, red where it pushes the cube further over. Where red below meets blue above, the cube rests stably, drawn thick; the other way round it balances unstably, drawn thin. The cubes above rest at the white dots, from face up through tilted to edge up and back.

## 5. Released from a heel

The hull is released from a heel of 30°, once with its centre of gravity 10 cm below the waterline and once 20 cm above it. It turns about its axis of heel, floating at each moment at the height where its buoyancy carries its weight; the righting couple, over the hull's moment of inertia about that axis, speeds its roll up or slows it down, and a little damping takes energy out. The roll and its rate are twists: turns about the axis of heel, the axis scaled by the angle and by the rate. The cell that releases the hull takes about seven seconds, most of it drawing the frames.

In [ ]:
# Which of the loads the released hull carries, its heel at release, its radius of gyration about the axis of
# heel as a share of the beam, the damping of its roll per second, the time step in seconds, the frames, the
# steps per frame, and the duration of a frame.
released, release_heel = [1, 4], 30.0
gyration, damping = 0.4, 0.4
step, n_frames, n_steps_per_frame, duration_ms = 0.05, 80, 3, 50

In [ ]:
def rolling(hull: Point, weight: Scalar, centres: Point) -> Iterator[tuple[Twist, Motor, Forque, Forque]]:
    """The hull released from a heel with each load: at every frame its roll, its placement in the world, its
    buoyancy and its weight."""
    inertia = weight / gravity * (gyration * beam) ** 2                   # [] Scalar+
    roll = mv.yz * np.full(centres.shape[0], np.radians(release_heel))    # [loads] Twist+
    roll_rate = mv.yz * np.zeros(centres.shape[0])                        # [loads] Twist+
    turns = (roll * (-1 / 2)).exp()                                       # [loads] Motor+
    heights = afloat(hull, turns, weight, mv.scalar(np.zeros((centres.shape[0], 1))), n_newton_steps)
    push, weighing = forces(hull, turns, heights, weight, centres)        # [loads] Forque-, Forque-
    for _ in range(n_frames):
        yield roll, placement(turns, heights), push, weighing
        for _ in range(n_steps_per_frame):
            # The righting couple about the axis of heel turns the roll, against its damping.
            roll_rate = roll_rate + (mv.yz * (((push + weighing) & heel_axis) / inertia) - damping * roll_rate) * step
            roll = roll + roll_rate * step
            turns = (roll * (-1 / 2)).exp()
            # The previous height is a close start: one Newton step keeps the hull afloat.
            heights = afloat(hull, turns, weight, heights, 1)
            push, weighing = forces(hull, turns, heights, weight, centres)


animated, rolled = tee(rolling(hull, weight, centres[released]))
show_animation(render.animate_release(hull, centres[released], animated, heels, couples[:, released], weight),
               "floating_release", duration_ms)
frames = list(rolled)

With its centre of gravity low the hull rocks back and settles upright; with it high the hull rolls over and settles upside down. On the right, each load's righting arm curve, continued past half a turn, carries the hull's heel and arm so far.

In [ ]:
# checks
# The slope of each righting arm curve at upright is the metacentric height against heel.
arms = (-(couples & heel_axis) / weight).to_array()                      # [heels, loads]
assert np.abs(arms[1] / np.sin(np.radians(heels[1])) - (stiffnesses(mv.yz, mv.yz) / weight).to_array()).max() < 1e-2
# A cube floats face up only below a density of (3 - sqrt(3)) / 6 or above (3 + sqrt(3)) / 6.
face_up = np.abs(stable_heels(cube_couples)) < 1e-6
assert np.array_equal(face_up, (densities < (3 - np.sqrt(3)) / 6) | (densities > (3 + np.sqrt(3)) / 6))
# The hull with the low load comes back upright; with the high one it capsizes and floats upside down.
low, high = np.degrees((frames[-1][0] & heel_axis.dual()).to_array())
assert abs(low) < 5 and abs(high - 180) < 10